### GPU Configuration testing

In [1]:
import sys
import platform
import torch
import subprocess

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this experiment.")

print(f"GPU count: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)

    print(f"\nGPU {i}:")
    print(f"  Name: {props.name}")
    print(f"  VRAM: {props.total_memory / 1024**3:.2f} GiB")
    print(f"  Compute capability: {props.major}.{props.minor}")

if not torch.cuda.is_bf16_supported():
    raise RuntimeError("BF16 support is required for the selected training configuration.")

print(f"\nBF16 supported: {torch.cuda.is_bf16_supported()}")

x = torch.randn(64, 64, device="cuda")
y = x @ x

print(f"CUDA test: {y.shape}")

Python: 3.12.13
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU count: 2

GPU 0:
  Name: Tesla T4
  VRAM: 14.56 GiB
  Compute capability: 7.5

GPU 1:
  Name: Tesla T4
  VRAM: 14.56 GiB
  Compute capability: 7.5

BF16 supported: True
CUDA test: torch.Size([64, 64])


In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
subprocess.check_call([sys.executable, "-m","pip","install","-q","--no-deps","torchao>=0.16.0"])
!pip install -q liger-kernel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.4/506.4 kB 3.5 MB/s eta 0:00:00


In [4]:
import json
import math
import random
from pathlib import Path

import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_cosine_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [5]:
SEED = 42

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [6]:
# Model config
MODEL_NAME = "Qwen/Qwen2.5-1.5B"
MAX_LENGTH = 2048

# LoRA config
LORA_R = 128
LORA_ALPHA = 512
LORA_DROPOUT = 0.1

LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
]

# Training Parameters
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 4

MICRO_BATCH_SIZE = 1
EFFECTIVE_BATCH_SIZE = 128
GRAD_ACCUM_STEPS = EFFECTIVE_BATCH_SIZE // MICRO_BATCH_SIZE
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0

# Memory parameters
DTYPE = torch.bfloat16
USE_GRADIENT_CHECKPOINTING = True

## WarmUP data fraction was take 5% of total data
WARMUP_FRACTION = 0.05

# Experiment directories
OUTPUT_ROOT = Path("./less_phase1")

CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
LOG_ROOT = OUTPUT_ROOT / "logs"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

In [7]:
from datasets import load_dataset, concatenate_datasets

DATA_FILES = {
    "flan_v2": Path("/kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/flan_v2_mini.jsonl"),
    "cot": Path("/kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/cot_mini.jsonl"),
    "dolly": Path("/kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/dolly_mini.jsonl"),
    "oasst1": Path("/kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/oasst1_mini.jsonl"),
}

datasets = {}

for name, path in DATA_FILES.items():
    ds = load_dataset(
        "json",
        data_files=str(path),
        split="train",
    )
    datasets[name] = ds
    print(f"{name:10s}: {len(ds):,} entries")


def merge_datasets(dataset_dict):
    return concatenate_datasets(list(dataset_dict.values()))


candidate_dataset = merge_datasets(datasets)
print(f"\nMerged dataset: {len(candidate_dataset):,} entries")

Generating train split: 0 examples [00:00, ? examples/s]

flan_v2   : 3,000 entries


Generating train split: 0 examples [00:00, ? examples/s]

cot       : 3,000 entries


Generating train split: 0 examples [00:00, ? examples/s]

dolly     : 1,500 entries


Generating train split: 0 examples [00:00, ? examples/s]

oasst1    : 1,500 entries

Merged dataset: 9,000 entries


In [8]:
warmup_size = int(len(candidate_dataset) * WARMUP_FRACTION)

rng = np.random.default_rng(SEED)
warmup_indices = rng.choice(
    len(candidate_dataset),
    size=warmup_size,
    replace=False,
)

warmup_dataset = candidate_dataset.select(warmup_indices)

print(f"Candidate dataset: {len(candidate_dataset):,} entries")
print(f"Warm-up dataset:   {len(warmup_dataset):,} entries")

Candidate dataset: 9,000 entries
Warm-up dataset:   450 entries


In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print(f"Tokenizer: {MODEL_NAME}")
print(f"Vocabulary size: {len(tokenizer):,}")


# Tokenize warm-up dataset
def tokenize_example(example):
    encoded = tokenizer.apply_chat_template(example["messages"], tokenize=True, add_generation_prompt=False)

    if hasattr(encoded, "encodings") and encoded.encodings:
        input_ids = encoded.encodings[0].ids
    elif hasattr(encoded, "input_ids"):
        input_ids = encoded["input_ids"]
        if input_ids and isinstance(input_ids[0], list):
            input_ids = input_ids[0]
    else:
        input_ids = encoded

    input_ids = list(input_ids)[:MAX_LENGTH]

    return {
        "input_ids": input_ids,
        "labels": input_ids.copy(),
        "attention_mask": [1] * len(input_ids),
    }


warmup_tokenized = warmup_dataset.map(
    tokenize_example,remove_columns=warmup_dataset.column_names, desc="Tokenizing warm-up dataset"
)

print("Tokenized examples:", len(warmup_tokenized))
print("Columns:", warmup_tokenized.column_names)
print("First example token length:", len(warmup_tokenized[0]["input_ids"]))

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer: Qwen/Qwen2.5-1.5B
Vocabulary size: 151,665


Tokenizing warm-up dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

Tokenized examples: 450
Columns: ['input_ids', 'labels', 'attention_mask']
First example token length: 102


In [10]:
# Data collator
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False)

from torch.utils.data import DataLoader

train_loader = DataLoader(
    warmup_tokenized,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
)

In [11]:
MODEL_DEVICE = torch.device("cuda:0")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
    trust_remote_code=True,
)

model.to(MODEL_DEVICE)

print(f"Model: {MODEL_NAME}")
print(f"Device: {MODEL_DEVICE}")
print(f"Dtype: {next(model.parameters()).dtype}")
print(f"Total Parameters: {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model: Qwen/Qwen2.5-1.5B
Device: cuda:0
Dtype: torch.bfloat16
Total Parameters: 1,543,714,304


In [12]:
model.config.use_cache = False

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 34,865,152 || all params: 1,578,579,456 || trainable%: 2.2086


In [13]:
# GPU memory check
torch.cuda.synchronize(MODEL_DEVICE)

allocated_gb = torch.cuda.memory_allocated(MODEL_DEVICE) / (1024 ** 3)
reserved_gb = torch.cuda.memory_reserved(MODEL_DEVICE) / (1024 ** 3)
total_gb = torch.cuda.get_device_properties(MODEL_DEVICE).total_memory / (1024 ** 3)

print(f"GPU:              {torch.cuda.get_device_name(MODEL_DEVICE)}")
print(f"Total VRAM:       {total_gb:.2f} GiB")
print(f"Allocated:        {allocated_gb:.2f} GiB")
print(f"Reserved:         {reserved_gb:.2f} GiB")
print(f"Free (approx.):   {total_gb - reserved_gb:.2f} GiB")

GPU:              Tesla T4
Total VRAM:       14.56 GiB
Allocated:        3.01 GiB
Reserved:         3.24 GiB
Free (approx.):   11.32 GiB


In [14]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# Training step calculation
num_micro_batches = len(train_loader)
steps_per_epoch = math.ceil(num_micro_batches / GRAD_ACCUM_STEPS)
total_training_steps = (steps_per_epoch * NUM_EPOCHS)

print(f"Micro-batches per epoch:   {num_micro_batches:,}")
print(f"Gradient accumulation:     {GRAD_ACCUM_STEPS}")
print(f"Optimizer steps per epoch:  {steps_per_epoch:,}")
print(f"Number of epochs:           {NUM_EPOCHS}")
print(f"Total optimizer steps:      {total_training_steps:,}")

Micro-batches per epoch:   450
Gradient accumulation:     128
Optimizer steps per epoch:  4
Number of epochs:           4
Total optimizer steps:      16


In [15]:
# Learning-rate scheduler

from transformers import get_cosine_schedule_with_warmup

num_warmup_steps = int(WARMUP_RATIO * total_training_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=total_training_steps,
)

trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

optimizer_param_count = sum(
    p.numel() for group in optimizer.param_groups
    for p in group["params"]
)

trainable_param_count = sum(
    p.numel() for p in trainable_params
)

print(f"Trainable parameters:       {trainable_param_count:,}")
print(f"Optimizer parameters:       {optimizer_param_count:,}")

assert optimizer_param_count == trainable_param_count

Trainable parameters:       34,865,152
Optimizer parameters:       34,865,152


In [16]:
import torch

model.config.use_cache = False
model.enable_input_require_grads()

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

model.train()

print("Gradient checkpointing:", model.is_gradient_checkpointing)
print("use_cache:", model.config.use_cache)

print(
    "GPU allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB"
)
print(
    "GPU reserved:",
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GiB"
)

Gradient checkpointing: True
use_cache: False
GPU allocated: 3.01 GiB
GPU reserved: 3.24 GiB


In [17]:
import os
import gc
import numpy as np
import torch
from tqdm.auto import tqdm

model.config.use_cache = False
model.base_model.model.config._attn_implementation = "eager"

model.enable_input_require_grads()

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    }
)

from liger_kernel.transformers import LigerFusedLinearCrossEntropyLoss

fused_loss_fn = LigerFusedLinearCrossEntropyLoss(
    reduction="mean",
    ignore_index=-100,
)

model.train()

transformer = model.base_model.model.model
lm_head = model.base_model.model.lm_head

fused_loss_fn = LigerFusedLinearCrossEntropyLoss(
    reduction="mean",
    ignore_index=-100,
)

def get_lora_parameters(model):
    return [
        p for p in model.parameters()
        if p.requires_grad
    ]

os.makedirs("./checkpoints", exist_ok=True)

global_step = 0

for epoch in range(1, NUM_EPOCHS + 1):

    running_loss = 0.0
    epoch_lrs = []

    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{NUM_EPOCHS}",
    )

    for step, batch in enumerate(progress, start=1):

        batch = {
            k: v.to(
                MODEL_DEVICE,
                non_blocking=True
            )
            for k, v in batch.items()
        }

        labels = batch["labels"].reshape(-1)

        with torch.autocast(
            device_type="cuda",
            dtype=DTYPE,
        ):

            outputs = transformer(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                use_cache=False,
                return_dict=True,
            )

            hidden_states = outputs.last_hidden_state.reshape(
                -1,
                outputs.last_hidden_state.shape[-1],
            )

            loss = fused_loss_fn(
                lm_head.weight,
                hidden_states,
                labels,
            )

            loss_for_backward = (
                loss / GRAD_ACCUM_STEPS
            )

        loss_value = loss.detach().item()

        loss_for_backward.backward()

        running_loss += loss_value

        should_step = (
            step % GRAD_ACCUM_STEPS == 0
            or step == len(train_loader)
        )

        if should_step:

            torch.nn.utils.clip_grad_norm_(
                get_lora_parameters(model),
                max_norm=MAX_GRAD_NORM,
            )

            optimizer.step()
            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )

            global_step += 1

            current_lr = scheduler.get_last_lr()[0]
            epoch_lrs.append(current_lr)

            progress.set_postfix(
                loss=f"{loss_value:.4f}",
                lr=f"{current_lr:.3e}",
                step=global_step,
                gpu=f"{torch.cuda.memory_allocated() / 1024**3:.2f}G",
            )

        del batch
        del labels
        del outputs
        del hidden_states
        del loss
        del loss_for_backward

        gc.collect()

    avg_loss = running_loss / len(train_loader)

    avg_lr = (
        float(np.mean(epoch_lrs))
        if epoch_lrs
        else 0.0
    )

    print(
        f"\nEpoch {epoch}: "
        f"loss={avg_loss:.5f}, "
        f"lr={avg_lr:.6e}, "
        f"optimizer_steps={global_step}"
    )

    print(
        f"GPU allocated: "
        f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB"
    )

    print(
        f"GPU reserved: "
        f"{torch.cuda.memory_reserved() / 1024**3:.2f} GiB"
    )

    # Save progress after every epoch
    checkpoint_dir = f"./checkpoints/epoch_{epoch}"
    os.makedirs(checkpoint_dir, exist_ok=True)

    model.save_pretrained(checkpoint_dir)
    tokenizer.save_pretrained(checkpoint_dir)

    torch.save(
        {
            "epoch": epoch,
            "global_step": global_step,
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "loss": avg_loss,
        },
        f"{checkpoint_dir}/training_state.pt",
    )

    print(
        f"Checkpoint saved: {checkpoint_dir}"
    )

    gc.collect()
    torch.cuda.empty_cache()

print("\nTraining complete.")
print(
    f"Final optimizer step: {global_step}"
)

Epoch 1/4:   0%|          | 0/450 [00:00<?, ?it/s]


Epoch 1: loss=13.55785, lr=1.860810e-05, optimizer_steps=4
GPU allocated: 3.28 GiB
GPU reserved: 7.08 GiB
Checkpoint saved: ./checkpoints/epoch_1


Epoch 2/4:   0%|          | 0/450 [00:00<?, ?it/s]


Epoch 2: loss=11.15947, lr=1.283336e-05, optimizer_steps=8
GPU allocated: 3.28 GiB
GPU reserved: 6.05 GiB
Checkpoint saved: ./checkpoints/epoch_2


Epoch 3/4:   0%|          | 0/450 [00:00<?, ?it/s]


Epoch 3: loss=10.06712, lr=5.398873e-06, optimizer_steps=12
GPU allocated: 3.28 GiB
GPU reserved: 6.24 GiB
Checkpoint saved: ./checkpoints/epoch_3


Epoch 4/4:   0%|          | 0/450 [00:00<?, ?it/s]


Epoch 4: loss=9.53907, lr=6.596639e-07, optimizer_steps=16
GPU allocated: 3.28 GiB
GPU reserved: 5.45 GiB
Checkpoint saved: ./checkpoints/epoch_4

Training complete.
Final optimizer step: 16
